In [4]:
%load_ext autoreload
%autoreload 2
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset
import sys
sys.path.append('../../../src/benchmark/')
sys.path.append('../../../src/utils/')
from build_model import resnet50_
from utils import hdf5_dataset, list_to_dict, viz_dataloader
from prediction_analysis import confusion_matrix, plot_cm, prediction_vs_actual, most_confused

NAME = '01102024-benchmark-resnet50_from_scratch-v4_10m'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/home/yichen/anaconda3/envs/symmetry/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/yichen/anaconda3/envs/symmetry/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [3]:
symmetry_classes = ['p1', 'p2', 'pm', 'pg', 'cm', 'pmm', 'pmg', 'pgg', 'cmm', 'p4', 'p4m', 'p4g', 'p3', 'p3m1', 'p31m', 'p6', 'p6m']
label_converter = list_to_dict(symmetry_classes)

# imagenet
train_ds = hdf5_dataset('../../../datasets/imagenet_v4_rot_10m_train_unchunked.h5', folder='train', transform=transforms.ToTensor())
train_dl = DataLoader(train_ds, batch_size=230, shuffle=True, num_workers=2)
viz_dataloader(train_dl, label_converter=label_converter)

valid_ds = hdf5_dataset('../../../datasets/imagenet_v4_rot_2m_valid_unchunked.h5', folder='valid', transform=transforms.ToTensor())
valid_dl = DataLoader(valid_ds, batch_size=230, shuffle=False, num_workers=2)
viz_dataloader(valid_dl, label_converter=label_converter)

cv_atom_ds = hdf5_dataset('../../../datasets/atom_v4_rot_2m_unchunked.h5', folder='test', transform=transforms.ToTensor())
cv_atom_dl = DataLoader(cv_atom_ds, batch_size=230, shuffle=False, num_workers=2)
viz_dataloader(cv_atom_dl, label_converter=label_converter)

cv_noise_ds = hdf5_dataset('../../../datasets/noise_v4_rot_2m_unchunked.h5', folder='test', transform=transforms.ToTensor())
cv_noise_dl = DataLoader(cv_noise_ds, batch_size=230, shuffle=False, num_workers=2)
viz_dataloader(cv_noise_dl, label_converter=label_converter)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/scratch/yichen/imagenet_v4_rot_10m_train_unchunked.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
model = resnet50_(3, 17)
model = torch.load('../../../saved_models/05312023-benchmark-resnet50_from_scratch-v4_10m-epoch-20.pt')
device = torch.device('cuda:2')

### generate confusion matrix

In [ ]:
cm = confusion_matrix(model, train_dl, symmetry_classes, device, n_batches='all')
np.save(f'../../../saved_results/data_preprocessing/FFT_Radon_transform/{NAME}-train_cm.npy', cm)

cm = confusion_matrix(model, valid_dl, symmetry_classes, device, n_batches='all')
np.save(f'../../../saved_results/data_preprocessing/FFT_Radon_transform/{NAME}-valid_cm.npy', cm)

cm = confusion_matrix(model, cv_atom_dl, symmetry_classes, device, n_batches='all')
np.save(f'../../../saved_results/data_preprocessing/FFT_Radon_transform/{NAME}-cv_atom_cm.npy', cm)

cm = confusion_matrix(model, cv_noise_dl, symmetry_classes, device, n_batches='all')
np.save(f'../../../saved_results/data_preprocessing/FFT_Radon_transform/{NAME}-cv_noise_cm.npy', cm)

### visualize confusions 

In [ ]:
order = ['train', 'valid', 'cv_atom_cm', 'cv_noise_cm']
def sort_key(file_path):
    for index, key in enumerate(order):
        if key in file_path:
            return index
    return len(order)

files = glob.glob(f'../../../saved_results/Benchmark/ResNet50/*')
sorted_files = sorted(files, key=sort_key)
for file in sorted_files:
    cm = np.load(file)
    plot_cm(cm, symmetry_classes, title=None, cm_style='simple', fig_style='printing', font_size=4)

### visualize confusions in compact layout

In [ ]:
# print(sorted_files)
NAME = 'Summary_cm-' + '-'.join(os.path.basename(sorted_files[0]).split('-')[:2])
print(NAME)

fig, axes = plt.subplots(3, 1, figsize=(6.5, 13))
for i, (ax, file) in enumerate(zip(axes, sorted_files)):
    cm = np.load(file)
    plot_cm(cm, symmetry_classes, title=None, ax=ax, cm_style='simple', fig_style='printing', fig_index=i, font_size=4)
plt.tight_layout()
plt.savefig(f'../../../figures/Benchmark/ResNet50/{NAME}.png')
plt.savefig(f'../../../figures/data_preprocessing/FFT_Radon_transform/{NAME}.svg')
plt.show()